In [4]:
import cv2
import time
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision.hand_landmarker import (
    HandLandmarker,
    HandLandmarkerOptions,
    HandLandmarksConnections
)

In [5]:
print(dir(mp))

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [6]:
BaseOptions = mp.tasks.BaseOptions

VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult


options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2
    )

landmarker = HandLandmarker.create_from_options(options)
HAND_CONNECTIONS = HandConnections.HAND_CONNECTIONS

In [12]:
cap = cv2.VideoCapture(0)

prev_time = 0

while True:
    ret,img = cap.read()
    img = cv2.flip(img, 1)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    cv2.putText(img , f"FPS: {fps:.2f}",(50,100),cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)
    
    timestamp = int(time.time() * 1000)
    result = landmarker.detect_for_video(mp_image , timestamp)
    timestamp += 1

    
    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            h, w, _ = img.shape
            points = []

            # Convert normalized landmarks to pixel coordinates
            for lm in hand_landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                points.append((x, y))
                cv2.circle(img, (x, y), 4, (0, 255, 0), -1)
            
            # Draw lines between landmarks
            
            for conn in HAND_CONNECTIONS:
                start = conn.start
                end = conn.end
                cv2.line(img, points[start], points[end], (255, 0, 0), 3)
    
    
    
    cv2.imshow("Image", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
        

        
cap.release()
cv2.destroyAllWindows()